# 03 · Pair GraphCast activations with cyclone labels

**Choose an A100 GPU with high system RAM, connect the same Google Drive you used in notebooks 01–02, and select Runtime → Run all.** The default run has 28 global snapshots and saves about 2.35 GB of activations in `My Drive/AMS691_GraphCast/cyclones_2019_2021`. A disconnected run can resume from completed snapshots.

For one time such as **2019-02-07 06:00 UTC**, each of the 40,962 GraphCast mesh nodes gets a row with 512 layer-8 values and a binary human-derived label. The weather inputs are 00:00 and 06:00. The model predicts 12:00. We pair the internal activation with the cyclone mask at **06:00**, the last observed input. This measures whether the feature represents a cyclone present at initialization.

## 1. Connect Drive and load the labeled data

The project includes the labels already projected onto the exact mesh that notebook 01 used. The pinned code and data allow a fresh Colab session to reproduce the same experiment.

In [1]:
# @title Connect Drive and load the project
import json
from google.colab import drive

import os
import subprocess
import sys
from pathlib import Path

# One immutable code version is shared by all six notebooks.
RELEASE = '3cf6820dbb3f40e9c48f13fe9c77601e10d4c01c'
REPO = Path('/content') / ('ams691_project_' + RELEASE[:12])
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'init', '-q', str(REPO)], check=True)
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--verify', 'HEAD'],
                      capture_output=True, text=True)
if head.stdout.strip() != RELEASE:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', '--depth', '1',
                    'https://github.com/observerhan/graphcast-interpretability.git', RELEASE], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', RELEASE], check=True)
sys.path.insert(0, str(REPO / 'scripts'))
sys.path.insert(0, str(REPO / 'src'))

drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/AMS691_GraphCast')
LABEL_RUN = DRIVE_PROJECT / 'cyclones_2019_2021'
DATA = REPO / 'data' / 'cyclone_2019_2021'


Mounted at /content/drive


## 2. Review the annotated times and select the run size

The cleaned ClimateNetLarge release provides two binary masks per timestamp. A grid cell is positive if either annotator marked it. We treat each connected region at a single time as one footprint. Connections include diagonal neighbors and wrap across the longitude seam. A storm can contribute footprints on several dates.

The full-model ERA5 source used in notebook 01 has 00, 06, 12 and 18 UTC. We keep exact matches. Each mesh node gets 1 when **every** linked grid cell is inside the combined mask. A mixed boundary neighborhood gets 0. The vertex order and grid-to-mesh radius come from the pinned GraphCast code and checkpoint.

In [2]:
# @title Annotated times in 2019–2021
import pandas as pd
from IPython.display import display
from cyclone_data import read_labels, read_samples

inventory = pd.read_csv(DATA / 'mask_inventory.csv')
def counts(group):
    return group.groupby('year').agg(
        snapshots=('center', 'size'),
        snapshots_with_TC=('union_regions', lambda regions: (regions > 0).sum()),
        cyclone_footprints=('union_regions', 'sum'),
    )
display(pd.concat({'All annotated hours': counts(inventory),
                   'Exact GraphCast hours': counts(inventory[inventory.six_hourly])}, axis=1))
matched = {sid for field in inventory.matched_track_sids.dropna() for sid in field.split()}
matched_regions = int(inventory.center_matched_regions.sum())
print(f"{len(matched)} distinct NOAA IBTrACS storm IDs have a center inside a marking.")
print(f"{matched_regions} of {inventory.union_regions.sum()} footprints contain a matched center.")

All annotated hours                                       \
               snapshots snapshots_with_TC cyclone_footprints   
year                                                            
2019                 253               173                284   
2020                 231               139                239   
2021                 213               140                246   

     Exact GraphCast hours                                       
                 snapshots snapshots_with_TC cyclone_footprints  
year                                                             
2019                    39                28                 45  
2020                    44                27                 43  
2021                    34                22                 37

206 distinct NOAA IBTrACS storm IDs have a center inside a marking.
671 of 769 footprints contain a matched center.


In [3]:
# @title Pick the labeled run
COHORT = 'course'  # @param ["course", "all_guarded"]
samples = read_samples(DATA, COHORT)
labels = read_labels(DATA / 'mesh_labels.npz', samples)
schedule = pd.DataFrame(samples)
summary = schedule.groupby('split', sort=False).agg(
    snapshots=('center', 'size'),
    maps_with_TC=('positive_nodes', lambda n: (n > 0).sum()),
    cyclone_nodes=('positive_nodes', 'sum'),
)
display(summary)
display(schedule.rename(columns={'center': 'forecast initialization (UTC)'}))
size_gb = len(samples) * 40962 * 512 * 4 / 1e9
print(f'{len(samples)} runs, about {size_gb:.2f} GB of float32 activations.')

,snapshots,maps_with_TC,cyclone_nodes
split,,,
fit,12,8,742
validation,8,5,548
test,8,5,560


,forecast initialization (UTC),split,positive_nodes
0,2019-02-07T06,fit,96
1,2019-02-27T00,fit,109
2,2019-04-07T18,fit,44
3,2019-04-17T00,fit,0
4,2019-04-29T00,fit,98
5,2019-05-15T18,fit,0
6,2019-06-14T18,fit,88
7,2019-07-20T18,fit,0
8,2019-08-05T18,fit,91
9,2019-09-19T12,fit,187


28 runs, about 2.35 GB of float32 activations.


The course run uses 12 snapshots from 2019 for fitting, 8 from 2020 for choosing the feature and cutoff, and 8 from 2021 for the final test. These include 4, 3 and 3 all-background maps respectively so a false alarm on a quiet map counts. Within each year and class of map we spaced the selections across the available dates before looking at model activations. The larger `all_guarded` setting uses all 93 exact-hour snapshots from February through November over these years (about 7.8 GB). Both settings have a December–January gap between groups. We checked all active NOAA main-track storm IDs across the three groups, including storms spanning multiple days.

## 3. Prepare the GraphCast runtime

The repository script `scripts/setup_graphcast.py` prepares the separate GraphCast environment. The collector imports its extraction functions directly from `scripts/ams691_extract.py`.


In [4]:
# @title Prepare the same full GraphCast used for SAE training
subprocess.run([sys.executable, '-u', str(REPO / 'scripts' / 'setup_graphcast.py')], check=True)


CompletedProcess(args=['/usr/bin/python3', '-u', '/content/ams691_project_b00fd40e7505/scripts/setup_graphcast.py'], returncode=0)

## 4. Run GraphCast at the listed hours

Notebook 01 used noon for its monthly samples. This run keeps the hour in each mask filename, including when the input window crosses midnight. It uses the full 0.25° / 37-pressure-level model and the layer-8 post-residual hook. Each finished activation is checked and saved immediately.

In [5]:
# @title Extract layer 8 for these exact times
import signal

graphcast_python = '/content/ams691_graphcast_env/bin/python'
command = [graphcast_python, '-u', str(REPO / 'scripts' / 'collect_cyclone_activations.py'),
           str(LABEL_RUN), str(DATA), COHORT]
env = {**os.environ, 'PYTHONPATH': os.pathsep.join([str(REPO / 'scripts'), str(REPO / 'src'), os.environ.get('PYTHONPATH', '')]),
       'XLA_PYTHON_CLIENT_PREALLOCATE': 'false'}
process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('Extraction stopped. The last message above identifies the problem.')
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    process.wait()
    raise

All 28 labeled snapshots are ready in /content/drive/MyDrive/AMS691_GraphCast/cyclones_2019_2021/course.


## 5. Check one labeled example

Row `i` in the activation file pairs with entry `i` in that timestamp’s label vector. The following cell chooses one marked node and shows its label and the first five values of its 512-dimensional activation.

In [6]:
# @title Verify an activation and its label
import numpy as np
from cyclone_data import activation_path, sha256

manifest = json.loads((LABEL_RUN / COHORT / 'dataset.json').read_text())
assert set(manifest['completed']) == set(schedule.center)
example = next(s for s in samples if s['positive_nodes'] > 0)
t = example['center']
path = activation_path(LABEL_RUN, t)
assert sha256(path) == manifest['completed'][t]
x = np.load(path, mmap_mode='r', allow_pickle=False)
i = int(np.flatnonzero(labels[t])[0])
print(f'{t} UTC | mesh row {i} | cyclone label {int(labels[t][i])}')
print('First five activation values:', x[i, :5])
print(f'{len(samples)} labeled snapshots are ready for notebook 04.')

2019-02-07T06 UTC | mesh row 345 | cyclone label 1
First five activation values: [ 0.94921875 -0.17773438 -0.3671875  -0.82421875  0.45507812]
28 labeled snapshots are ready for notebook 04.


The outputs are in `AMS691_GraphCast/cyclones_2019_2021`. Notebook 04 reads the selected time list, mesh labels and activations from this folder. The same pairs can be reused for a later KAN-SAE comparison.

[Cleaned ClimateNetLarge archive](https://dataverse.harvard.edu/api/access/datafile/11212451) · [Dataset methods](https://doi.org/10.1038/s41597-025-05480-0) · [NOAA IBTrACS](https://www.ncei.noaa.gov/products/international-best-track-archive) · [GraphCast single-feature probing, §2.5](https://arxiv.org/html/2512.24440v1#S2.SS5)